<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            border-left: 6px solid #28a745;
            padding: 30px;
            border-radius: 12px;
            margin: 25px 0;
            box-shadow: 0 8px 20px rgba(40,167,69,0.25);">
    <h1 style="margin: 0 0 15px 0; color: #ffffff; font-size: 2em; font-weight: 700;">
        ✅ SOLUTIONS - Part 1: Semantic Search Foundations with pgvector
    </h1>
    <h3 style="margin: 0 0 20px 0; color: #ffffff; opacity: 0.95; font-weight: 400; font-size: 1.15em;">
        Complete Reference Implementation
    </h3>
    <div style="background: rgba(255,255,255,0.08);
                backdrop-filter: blur(8px);
                border: 1px solid rgba(255,255,255,0.15);
                padding: 20px;
                border-radius: 8px;
                margin-top: 20px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.15);">
        <p style="margin: 0 0 12px 0; font-size: 1.1em; color: #ffffff;">
            <strong style="color: #ffffff;">🚀 Quick Start:</strong> To see the complete implementation:
        </p>
        <ol style="margin: 10px 0; padding-left: 25px; line-height: 1.8; font-size: 1.05em; color: #ffffff;">
            <li><strong>Select Kernel (Python 3.13) → Menu → Run All</strong> (or press Shift+Enter through each cell)</li>
            <li>Watch as the notebook executes all setup and solution code</li>
            <li>Review the completed exercise solution with explanatory comments</li>
        </ol>
        <p style="margin: 15px 0 0 0; font-size: 1.02em; color: #ffffff;">
            ⏱️ <strong style="color: #ffffff;">Total Runtime:</strong> ~1-2 minutes
        </p>
    </div>
</div>

---

## ⚙️ Step 0: Select Python Kernel

<div style="background: linear-gradient(135deg, rgba(245,158,11,0.18) 0%, rgba(217,119,6,0.18) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(245,158,11,0.35);
            border-left: 6px solid #f59e0b;
            padding: 20px;
            border-radius: 10px;
            margin: 20px 0;
            box-shadow: 0 4px 16px rgba(245,158,11,0.25);">
    <h4 style="margin: 0 0 15px 0; color: #fbbf24; font-size: 1.2em; font-weight: 600;">
        ⚠️ IMPORTANT FIRST STEP
    </h4>
    <p style="margin: 0; color: #e9ecef; line-height: 1.6; font-size: 1.02em;">
        Before running any code, ensure you're using <strong style="color: #fbbf24;">Python 3.13</strong> kernel.
    </p>
</div>

In [ ]:
# Verify Python Environment
import sys

python_version = sys.version.split()[0]
required_version = "3.13"

print(f"🐍 Python Version: {python_version}")
print("─" * 50)

if python_version.startswith(required_version):
    print("✅ Environment Verified")
    print(f"   Running Python {python_version} (Required: {required_version}.x)")
    print("   Ready to proceed with workshop exercises.")
else:
    print(f"⚠️  Environment Mismatch Detected")
    print(f"   Current: Python {python_version}")
    print(f"   Required: Python {required_version}.x")
    print("\n📋 Action Required:")
    print("   1. Click the kernel selector (top-right corner)")
    print("   2. Select 'Python 3.13' from the dropdown")
    print("   3. Re-run this cell to verify")

---

# Setup: Database Connection and Embedding Function

These cells establish the connection to Aurora PostgreSQL and define the embedding generation function needed for the solution.

In [ ]:
# ============================================================================
# Environment Setup: Database Connection & Bedrock Client Initialization
# ============================================================================

import os
import json
import psycopg
from psycopg.rows import dict_row
import boto3
from typing import List, Dict, Optional
import time
from dotenv import load_dotenv

# Load environment configuration
load_dotenv('/workshop/sample-dat406-build-agentic-ai-powered-search-apg/.env')

# ============================================================================
# Database Configuration
# ============================================================================

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'postgres')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Construct PostgreSQL connection string
conn_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# ============================================================================
# Amazon Bedrock Client Initialization
# ============================================================================

bedrock = boto3.client(
    'bedrock-runtime', 
    region_name=os.environ.get('AWS_REGION', 'us-west-2')
)

# ============================================================================
# Verification
# ============================================================================

print("=" * 60)
print("✅ Environment Configuration Complete")
print("=" * 60)
print(f"📊 Aurora Endpoint: {conn_string.split('@')[1].split('/')[0]}")
print(f"🤖 Bedrock Region:  {bedrock.meta.region_name}")
print(f"🗄️ Database Name:   {DB_NAME}")
print("=" * 60)

In [ ]:
def generate_embedding(text: str) -> List[float]:
    """
    Generate semantic embeddings using Amazon Titan Text Embeddings V2.
    
    Converts natural language text into a 1024-dimensional vector representation
    that captures semantic meaning, enabling similarity-based search beyond
    keyword matching.
    
    Args:
        text: Input text to embed (query or document content)
        
    Returns:
        List of 1024 floating-point values representing the semantic embedding
    """
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": text})
    )
    
    result = json.loads(response['body'].read())
    return result['embedding']


# ============================================================================
# Embedding Generation Test
# ============================================================================

sample_query = "wireless headphones for running"
sample_embedding = generate_embedding(sample_query)

print("=" * 60)
print("✅ Embedding Generation Successful")
print("=" * 60)
print(f"Query Text:        '{sample_query}'")
print(f"Vector Dimensions: {len(sample_embedding)}")
print(f"Sample Values:     {[f'{v:.6f}' for v in sample_embedding[:5]]}")
print(f"Value Range:       [{min(sample_embedding):.6f}, {max(sample_embedding):.6f}]")
print("=" * 60)

# 2C. Hands-On Exercise Solution

<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            border-left: 6px solid #28a745;
            padding: 25px;
            border-radius: 12px;
            margin: 20px 0;
            box-shadow: 0 8px 20px rgba(40,167,69,0.25);">
    <h2 style="margin: 0 0 15px 0; color: #ffffff; font-weight: 600;">
        ✅ Complete Implementation
    </h2>
    <p style="margin: 0; color: #ffffff; font-size: 1.1em; line-height: 1.6; opacity: 0.95;">
        Here's the complete solution for the semantic search exercise from Section 2C.
    </p>
</div>

In [ ]:
def semantic_search_with_filter(query_text: str, max_price: Optional[float] = None, limit: int = 5) -> List[Dict]:
    """
    Perform semantic search with optional price filtering.
    
    Demonstrates pgvector 0.8.0's automatic iterative scanning when combining
    vector similarity with business filters.
    
    Args:
        query_text: Natural language search query
        max_price: Optional maximum price filter
        limit: Number of results to return
        
    Returns:
        List of product dictionaries with similarity scores
    """
    
    # STEP 1: Generate query embedding
    # Convert natural language to 1024-dimensional vector for semantic comparison
    query_embedding = generate_embedding(query_text)
    
    
    # STEP 2: Build dynamic SQL query
    with psycopg.connect(conn_string) as conn:
        with conn.cursor(row_factory=dict_row) as cur:
            
            # Base WHERE clause and parameters
            where_clauses = ["embedding IS NOT NULL"]
            params = [query_embedding, query_embedding]  # For distance & similarity calculations
            
            # Add price filter if specified (enables pgvector 0.8.0 iterative scanning)
            if max_price is not None:
                where_clauses.append("price <= %s")
                params.append(max_price)
            
            where_clause = " AND ".join(where_clauses)
            
            # STEP 3: Execute semantic search with pgvector
            # <=> is cosine distance (0=identical, lower=more similar)
            # HNSW index accelerates vector search to sub-100ms
            query = f"""
                SELECT 
                    "productId",
                    product_description as title,
                    price,
                    stars,
                    reviews,
                    "category_name" as category,
                    embedding <=> %s::vector AS distance,
                    1 - (embedding <=> %s::vector) AS similarity
                FROM bedrock_integration.product_catalog
                WHERE {where_clause}
                ORDER BY embedding <=> %s::vector ASC
                LIMIT %s
            """
            
            # STEP 4: Add ORDER BY and LIMIT parameters
            # query_embedding appears 3x: distance, similarity, ORDER BY
            params.extend([query_embedding, limit])
            
            cur.execute(query, params)
            return [dict(row) for row in cur.fetchall()]

print("✅ Solution function defined")

## Testing the Solution

Let's verify the implementation works correctly with two test cases.

In [ ]:
# ============================================================================
# Verification: Test Your Implementation
# ============================================================================

print("🧪 Testing semantic search implementation...\n")

# Test 1: Basic semantic search
print("Test 1: 'wireless headphones for running' (no filter)")
print("-" * 60)

try:
    results = semantic_search_with_filter("wireless headphones for running", limit=3)
    
    if results and all(k in results[0] for k in ['title', 'price', 'similarity']):
        for idx, p in enumerate(results, 1):
            print(f"{idx}. {p['title'][:60]}")
            print(f"   ${p['price']:.2f} | {p['stars']}⭐ | {p['similarity']:.1%} match\n")
        print("✅ Test 1 PASSED\n")
    else:
        print("❌ Test 1 FAILED - Missing required fields\n")
except Exception as e:
    print(f"❌ Test 1 FAILED - {type(e).__name__}: {e}\n")

# Test 2: Semantic search with price filter
print("Test 2: 'something to keep coffee hot' (max $30)")
print("-" * 60)

try:
    results = semantic_search_with_filter("something to keep coffee hot", max_price=30.0, limit=3)
    
    if results:
        for idx, p in enumerate(results, 1):
            status = "✓" if p['price'] <= 30 else "⚠️"
            print(f"{idx}. {p['title'][:60]}")
            print(f"   ${p['price']:.2f} {status} | {p['stars']}⭐ | {p['similarity']:.1%} match\n")
        
        if all(p['price'] <= 30.0 for p in results):
            print("✅ Test 2 PASSED - Price filter working!\n")
        else:
            print("❌ Test 2 FAILED - Results exceed price limit\n")
    else:
        print("❌ Test 2 FAILED - No results\n")
except Exception as e:
    print(f"❌ Test 2 FAILED - {type(e).__name__}: {e}\n")

print("=" * 60)
print("🎉 If both tests passed, you've successfully built:")
print("   • Semantic search with pgvector <=> operator")
print("   • Dynamic SQL with business filters")
print("   • pgvector 0.8.0 iterative scanning")
print("=" * 60)

<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            padding: 20px;
            border-radius: 10px;
            margin: 30px 0;
            text-align: center;
            box-shadow: 0 6px 16px rgba(40,167,69,0.25);">
    <h3 style="margin: 0 0 10px 0; color: #ffffff; font-size: 1.4em; font-weight: 600;">
        🎉 Solution Complete!
    </h3>
    <p style="margin: 0; color: #ffffff; font-size: 1.05em; line-height: 1.6; opacity: 0.95;">
        You've mastered semantic search with filtering.<br>
        <strong style="color: #ffffff;">Ready for Part 2?</strong> Build custom agent tools that leverage this foundation!
    </p>
</div>

---

<div style="text-align: center; color: #888; font-size: 0.9em; margin-top: 30px;">
Built for AWS re:Invent 2025 | DAT406 Workshop
</div>